In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import tensorflow 
from tensorflow import keras
from keras.models import Sequential 
from keras.layers import Dense,Dropout, BatchNormalization 
from keras.regularizers import l1,l2
from keras.callbacks import EarlyStopping 
import seaborn as sns 
from mlxtend.plotting import plot_decision_regions 
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.model_selection import train_test_split as tts 
from keras.optimizers import Adam, RMSprop, SGD, Adagrad


In [2]:
df = pd.read_csv('diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
df.corr()['Outcome']

Pregnancies                 0.221898
Glucose                     0.466581
BloodPressure               0.065068
SkinThickness               0.074752
Insulin                     0.130548
BMI                         0.292695
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

In [6]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values


In [7]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [9]:
X.shape

(768, 8)

In [10]:
X_train,X_test,y_train,y_test = tts(X,y,test_size=0.2,random_state=1)

In [12]:
model = Sequential()

model.add(Dense(128,activation='relu',input_dim=8))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='adam',loss='binary_crossentropy', metrics=['accuracy'])

hist = model.fit(X_train,y_train,batch_size=32,epochs=10,validation_data=(X_test,y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.4870 - loss: 0.7068 - val_accuracy: 0.6558 - val_loss: 0.6514
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.7101 - loss: 0.6118 - val_accuracy: 0.7338 - val_loss: 0.5775
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7476 - loss: 0.5561 - val_accuracy: 0.7597 - val_loss: 0.5345
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7590 - loss: 0.5196 - val_accuracy: 0.7727 - val_loss: 0.5080
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7736 - loss: 0.4964 - val_accuracy: 0.7727 - val_loss: 0.4902
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7736 - loss: 0.4809 - val_accuracy: 0.7662 - val_loss: 0.4817
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7769 - loss: 0.4704 - val_accuracy: 0.7727 - val_loss: 0.4737
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7752 - loss: 0.4648 - val_accuracy: 0.7792 - v

In [13]:
# Here keras tuner automate the process which wil give results for different optimizers, activation function,etc

In [14]:
# How to select appropriate optimizers
# No. of Nodes in a Layer
# How to select of number of Layers
# all in one model

In [16]:
import keras_tuner as kt

In [17]:
def build_model(hp):
    # build model from scratch
    model = Sequential()
    model.add(Dense(32,activation='relu',input_dim=8))
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer=hp.Choice('optimizer',values=['adam','rmsprop','sgd','adadelta']),loss='binary_crossentropy',metrics=['accuracy'])

    return model




In [18]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy', max_trials=5)

c:\Users\rudra\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [19]:
tuner.search(X_train,y_train,epochs=10,validation_data=(X_test,y_test))

Trial 4 Complete [00h 00m 07s]
val_accuracy: 0.8051947951316833

Best val_accuracy So Far: 0.8116883039474487
Total elapsed time: 00h 00m 34s


In [23]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [25]:
model = tuner.get_best_models(num_models=1)[0]

In [26]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [27]:
model.fit(X_train,y_train,epochs=100,batch_size=32,initial_epoch=11,validation_data=(X_test,y_test))

Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - accuracy: 0.7720 - loss: 0.5118 - val_accuracy: 0.8182 - val_loss: 0.4872
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7769 - loss: 0.4983 - val_accuracy: 0.8052 - val_loss: 0.4764
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7720 - loss: 0.4889 - val_accuracy: 0.8052 - val_loss: 0.4677
Epoch 15/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7752 - loss: 0.4811 - val_accuracy: 0.8052 - val_loss: 0.4627
Epoch 16/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7736 - loss: 0.4758 - val_accuracy: 0.8117 - val_loss: 0.4575
Epoch 17/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7752 - loss: 0.4703 - val_accuracy: 0.8117 - val_loss: 0.4549
Epoch 18/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7801 - loss: 0.4664 - val_accuracy: 0.8052 - val_loss: 0.4516
Epoch 19/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7752 - loss: 0.4628 - val_accu

In [39]:
#  hyperparmeter tuning on no. of nodes in a particular layer
def build_model(hp):

    model =Sequential()
    units = hp.Int('units',8,128,step=8)
    model.add(Dense(units=units, activation='relu',input_dim=8))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam',loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [40]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy', max_trials=5,directory='mydir')

In [41]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 06s]
val_accuracy: 0.7792207598686218

Best val_accuracy So Far: 0.8051947951316833
Total elapsed time: 00h 00m 32s


In [42]:
tuner.get_best_hyperparameters()[0].values

{'units': 128}

In [45]:
model1 = tuner.get_best_models(num_models=1)[0]

c:\Users\rudra\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\rudra\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [46]:
model1.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,281 (5.00 KB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 0 (0.00 B)

In [49]:
# How to select number of layers 
def build_model(hp):
    model = Sequential()
    model.add(Dense(128,activation='relu',input_dim=8))
    for i in range(hp.Int('num_layers',min_value=0, max_value=10)):
        model.add(Dense(128,activation='relu'))
    
    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [50]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=10,directory='mydir1',project_name='num_layers_tuner')


c:\Users\rudra\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [51]:
tuner.search(X_train,y_train,epochs=10,validation_data=(X_test,y_test))

Trial 10 Complete [00h 00m 21s]
val_accuracy: 0.8246753215789795

Best val_accuracy So Far: 0.8246753215789795
Total elapsed time: 00h 02m 51s


In [52]:
tuner.get_best_hyperparameters()[0].values


{'num_layers': 3}

In [53]:
model = tuner.get_best_models(num_models=1)[0]

c:\Users\rudra\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\rudra\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [54]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,817 (198.50 KB)

 Trainable params: 50,817 (198.50 KB)

 Non-trainable params: 0 (0.00 B)

In [57]:
model.fit(X_train,y_train,epochs=100,initial_epoch=11,validation_data=(X_test,y_test))

Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 65ms/step - accuracy: 0.7932 - loss: 0.4080 - val_accuracy: 0.7922 - val_loss: 0.4781
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8274 - loss: 0.3895 - val_accuracy: 0.7987 - val_loss: 0.4871
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8225 - loss: 0.3759 - val_accuracy: 0.8182 - val_loss: 0.4991
Epoch 15/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8436 - loss: 0.3598 - val_accuracy: 0.8182 - val_loss: 0.4961
Epoch 16/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8388 - loss: 0.3577 - val_accuracy: 0.7922 - val_loss: 0.5096
Epoch 17/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8388 - loss: 0.3483 - val_accuracy: 0.8117 - val_loss: 0.5079
Epoch 18/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8436 - loss: 0.3342 - val_accuracy: 0.8117 - val_loss: 0.5229
Epoch 19/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.8697 - loss: 0.3124 - val_accu

In [70]:
def build_model(hp):
    model = Sequential()
    counter = 0
    
    for i in range(hp.Int('num_layers',min_value=1,max_value=10)):
        if counter==0:
            model.add(
                Dense(
                    units=hp.Int('units' + str(i), min_value=8,max_value=128,step=8),
                    activation= hp.Choice('activation'+str(i), values=['relu','sigmoid','tanh','elu']),
                    input_dim=8
                )
            )
            model.add(Dropout(hp.Choice('dropout'+str(i), values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
        else:
            model.add(
                Dense(
                    units=hp.Int('units' + str(i), min_value=8,max_value=128,step=8),
                    activation= hp.Choice('activation'+str(i), values=['relu','sigmoid','tanh','elu']),
                )
            )    
            model.add(Dropout(hp.Choice('dropout'+str(i), values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
        counter+=1    

    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer=hp.Choice('optimizers',values=['adam','rmsprop','sgd','nadam','adadelta']), loss='binary_crossentropy', metrics=['accuracy'])   

    return model   

In [71]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5,directory='mydir',project_name='final1')

In [73]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 09s]
val_accuracy: 0.8116883039474487

Best val_accuracy So Far: 0.8116883039474487
Total elapsed time: 00h 00m 51s


In [74]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 3,
 'units0': 56,
 'activation0': 'elu',
 'dropout0': 0.4,
 'optimizers': 'rmsprop',
 'units1': 112,
 'activation1': 'relu',
 'dropout1': 0.2,
 'units2': 32,
 'activation2': 'relu',
 'dropout2': 0.1,
 'units3': 40,
 'activation3': 'tanh',
 'dropout3': 0.9,
 'units4': 48,
 'activation4': 'tanh',
 'dropout4': 0.3,
 'units5': 8,
 'activation5': 'relu',
 'dropout5': 0.6,
 'units6': 56,
 'activation6': 'tanh',
 'dropout6': 0.6,
 'units7': 48,
 'activation7': 'elu',
 'dropout7': 0.2,
 'units8': 88,
 'activation8': 'tanh',
 'dropout8': 0.1}

In [75]:
model = tuner.get_best_models(num_models=1)[0]

c:\Users\rudra\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [78]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 56)             │           504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 56)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 112)            │         6,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 112)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         3,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,076 (82.33 KB)

 Trainable params: 10,537 (41.16 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 10,539 (41.17 KB)

In [77]:
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=200,initial_epoch=6)

Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 65ms/step - accuracy: 0.7687 - loss: 0.4745 - val_accuracy: 0.8117 - val_loss: 0.4943
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7655 - loss: 0.4776 - val_accuracy: 0.7857 - val_loss: 0.4977
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.7427 - loss: 0.4823 - val_accuracy: 0.7857 - val_loss: 0.4985
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7720 - loss: 0.4874 - val_accuracy: 0.8117 - val_loss: 0.4913
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7736 - loss: 0.4769 - val_accuracy: 0.8182 - val_loss: 0.4859
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7590 - loss: 0.4773 - val_accuracy: 0.8182 - val_loss: 0.4817
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7704 - loss: 0.4688 - val_accuracy: 0.8052 - val_loss: 0.4865
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7638 - loss: 0.4842 - val_accurac